# Fake News Detection Using NLP and Machine Learning
**Course:** CSE570 – Machine Learning with Python  
**Dataset:** `fake_news_dataset.csv`  
**Task:** Binary classification (`fake` / `real`)

> Run cells from top to bottom. Keep the CSV in the project root or update `DATA_PATH`. Do not report unmeasured results: use the metrics generated by your run.


## 1. Imports and configuration

In [ ]:
import re, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency, mannwhitneyu

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
DATA_PATH = Path('../fake_news_dataset.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('fake_news_dataset.csv')
df = pd.read_csv(DATA_PATH)
pd.set_option('display.max_columns', None)
print('Dataset shape:', df.shape)
df.head()


## 2. Problem statement and objectives

**Problem:** Classify a news article as fake or real from its headline and article text.  
**Target:** `label`.  
**Input features:** `title` and `text` (the primary text fields). Metadata columns are inspected during EDA but excluded from the initial text model to reduce source/author leakage.  
**Objectives:** (1) inspect dataset quality, (2) analyze text and label patterns, (3) train and compare at least three classifiers, (4) select a model using test metrics, and (5) package a prediction application.  
**Constraint:** This dataset is not a substitute for independent, verified real-world news data. Model performance only describes this dataset and split.

## 3. Dataset audit

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().to_frame('missing_values'))
print('Duplicate rows:', df.duplicated().sum())
display(df['label'].value_counts(dropna=False).to_frame('count'))
display((df['label'].value_counts(normalize=True)*100).round(2).to_frame('percentage'))
for col in ['source', 'category']:
    if col in df:
        print('\\n', col)
        display(df[col].value_counts().head(15).to_frame('count'))


## 4. Data cleaning and feature construction

In [ ]:
required = ['title', 'text', 'label']
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df = df.drop_duplicates().copy()
df = df.dropna(subset=required).copy()
for c in ['title', 'text', 'label']:
    df[c] = df[c].astype(str).str.strip()
df = df[(df['title'] != '') & (df['text'] != '') & (df['label'] != '')].copy()
df['label'] = df['label'].str.lower()
df = df[df['label'].isin(['fake', 'real'])].copy()

def clean_text(value):
    value = str(value).lower()
    value = re.sub(r'<[^>]+>', ' ', value)
    value = re.sub(r'http\\S+|www\\.\\S+', ' ', value)
    value = re.sub(r'[^a-z\\s]', ' ', value)
    value = re.sub(r'\\s+', ' ', value).strip()
    return value

df['combined_text'] = (df['title'] + ' ' + df['text']).map(clean_text)
df['text_length'] = df['text'].str.len()
print('Cleaned shape:', df.shape)
display(df[['title','label','text_length','combined_text']].head())


## 5. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='label')
plt.title('Class Distribution'); plt.xlabel('News label'); plt.ylabel('Number of articles')
plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4))
sns.histplot(data=df, x='text_length', hue='label', bins=35, kde=True, element='step')
plt.title('Article Text Length by Label'); plt.tight_layout(); plt.show()

if 'category' in df.columns:
    plt.figure(figsize=(10,5))
    sns.countplot(data=df, y='category', hue='label')
    plt.title('News Category vs Label'); plt.tight_layout(); plt.show()

if 'category' in df.columns:
    contingency = pd.crosstab(df['category'], df['label'])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    print(f'Chi-square test category vs label: chi2={chi2:.3f}, dof={dof}, p={p_value:.6g}')
    print('Interpretation: a small p-value indicates an association in this dataset, not causation.')

fake_len = df.loc[df.label=='fake', 'text_length']
real_len = df.loc[df.label=='real', 'text_length']
u_stat, mw_p = mannwhitneyu(fake_len, real_len, alternative='two-sided')
print(f'Mann–Whitney test for text length: U={u_stat:.1f}, p={mw_p:.6g}')
print('Interpretation: compare the p-value with 0.05 and inspect the distributions; statistical significance is not practical importance.')


## 6. Train/test split and leakage-safe pipeline

In [ ]:
X = df['combined_text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print('Train:', X_train.shape[0], 'Test:', X_test.shape[0])
print('Training label distribution:\\n', y_train.value_counts(normalize=True).round(3))


## 7. Baseline and three or more ML models

In [ ]:
models = {
    'Dummy baseline': DummyClassifier(strategy='most_frequent'),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1500, class_weight='balanced'),
    'Linear SVM': LinearSVC(class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=250, random_state=RANDOM_STATE,
                                             class_weight='balanced_subsample', n_jobs=-1)
}
pipelines = {
    name: Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=100000,
                                  sublinear_tf=True)),
        ('model', estimator)
    ])
    for name, estimator in models.items()
}
results = []
fitted = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    row = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, pos_label='fake', zero_division=0),
        'Recall': recall_score(y_test, pred, pos_label='fake', zero_division=0),
        'F1': f1_score(y_test, pred, pos_label='fake', zero_division=0)
    }
    if hasattr(pipe, 'predict_proba'):
        scores = pipe.predict_proba(X_test)[:, list(pipe.classes_).index('fake')]
        row['ROC_AUC'] = roc_auc_score((y_test=='fake').astype(int), scores)
    elif hasattr(pipe, 'decision_function'):
        row['ROC_AUC'] = roc_auc_score((y_test=='fake').astype(int), pipe.decision_function(X_test))
    else:
        row['ROC_AUC'] = np.nan
    results.append(row); fitted[name] = pipe
results_df = pd.DataFrame(results).sort_values('F1', ascending=False)
display(results_df.round(4))


## 8. Confusion matrix and detailed report

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = fitted[best_name]
best_pred = best_model.predict(X_test)
print('Selected by test F1 (exploratory selection):', best_name)
print(classification_report(y_test, best_pred, digits=4))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred, cmap='Blues')
plt.title(f'Confusion Matrix – {best_name}')
plt.tight_layout(); plt.show()

# If the selected estimator exposes scores, plot ROC.
if hasattr(best_model, 'predict_proba'):
    score = best_model.predict_proba(X_test)[:, list(best_model.classes_).index('fake')]
    RocCurveDisplay.from_predictions((y_test=='fake').astype(int), score)
    plt.title(f'ROC Curve – {best_name}'); plt.tight_layout(); plt.show()
elif hasattr(best_model, 'decision_function'):
    score = best_model.decision_function(X_test)
    RocCurveDisplay.from_predictions((y_test=='fake').astype(int), score)
    plt.title(f'ROC Curve – {best_name}'); plt.tight_layout(); plt.show()


## 9. Cross-validation and hyperparameter tuning

In [ ]:
# Cross-validation is performed only on the training partition.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    cv_rows.append({'Model': name, 'CV F1 macro mean': scores.mean(), 'CV std': scores.std()})
display(pd.DataFrame(cv_rows).sort_values('CV F1 macro mean', ascending=False).round(4))

# Tune Logistic Regression. TF-IDF is inside the pipeline, so each fold learns vocabulary
# from its training fold only.
lr_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, max_features=100000, sublinear_tf=True)),
    ('model', LogisticRegression(max_iter=1500, class_weight='balanced'))
])
param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2)],
    'tfidf__max_features': [50000, 100000],
    'model__C': [0.5, 1.0, 2.0]
}
search = GridSearchCV(lr_pipe, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1)
search.fit(X_train, y_train)
print('Best parameters:', search.best_params_)
print('Best CV macro-F1:', round(search.best_score_, 4))
tuned_model = search.best_estimator_
tuned_pred = tuned_model.predict(X_test)
print(classification_report(y_test, tuned_pred, digits=4))


## 10. Interpretation and limitations

In [ ]:
# Logistic Regression coefficient interpretation (if tuned model is used)
vec = tuned_model.named_steps['tfidf']
clf = tuned_model.named_steps['model']
if hasattr(clf, 'coef_'):
    feature_names = np.array(vec.get_feature_names_out())
    coefs = clf.coef_[0]
    print('Terms most associated with class:', clf.classes_[1])
    print('Top positive terms:', feature_names[np.argsort(coefs)[-15:][::-1]].tolist())
    print('Top negative terms:', feature_names[np.argsort(coefs)[:15]].tolist())

print('Dataset limitations to discuss:')
print('- Dataset may contain synthetic or non-representative text; validate its provenance.')
print('- Publisher/author/date can create shortcuts, so they are not used in the primary text-only model.')
print('- Random train/test split does not prove performance on future or unseen publishers.')
print('- Fake/real labels may be noisy; model output is not a fact-checking verdict.')


## 11. Save final model and test a custom article

In [ ]:
ARTIFACT_DIR = Path('../artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)
joblib.dump(tuned_model, ARTIFACT_DIR / 'fake_news_pipeline.joblib')
print('Saved:', ARTIFACT_DIR / 'fake_news_pipeline.joblib')

def predict_news(title, article_text):
    combined = clean_text(str(title) + ' ' + str(article_text))
    prediction = tuned_model.predict([combined])[0]
    return prediction

sample_title = 'Example headline to test'
sample_text = 'Paste a complete news article here for a model prediction.'
print('Prediction:', predict_news(sample_title, sample_text))


## 12. Final conclusion (complete after running)

Write your conclusion using your actual results:
- State the best model and its measured test metrics.
- Explain the most important EDA observations.
- Compare cross-validation and held-out test performance.
- Mention errors visible in the confusion matrix.
- State clearly that the system is an educational screening prototype, not an authoritative fact-checker.
